# Day 61 — Classification Evaluation Metrics

This notebook covers Limitations of Simple Accuracy, Accuracy Paradox, Confusion Matrix, True/False Positives and Negatives, Precision, Recall, F1-Score, and Scikit-Learn Classification Reports

## Validation goals
- Understand the limitations of simple accuracy when handling heavily imbalanced datasets and the accuracy paradox
- Define the four components of a confusion matrix: True Positives, True Negatives, False Positives (Type I), and False Negatives (Type II)
- Derive and analyze core metrics: Precision (positive prediction quality), Recall (sensitivity / actual positive coverage), and F1-Score (harmonic mean)
- Implement a classification evaluation pipeline in Python using Scikit-Learn (`confusion_matrix`, `classification_report`, `accuracy_score`)

In regression, evaluation is straightforward (e.g., how many dollars off was our house price prediction?). In classification, measuring performance is more nuanced. Relying solely on overall accuracy can lead to dangerous blind spots—especially when dealing with imbalanced datasets like fraud detection or rare disease diagnosis.

### 1. The Limitations of Simple Accuracy

Imagine you are building a model to detect credit card fraud. Out of 10,000 daily transactions in your dataset:
- 9,900 are legitimate transactions.
- 100 are fraudulent transactions ($1\%$).

If you write a lazy, useless program that simply outputs "Legitimate" for every single transaction without looking at the data, what is its Accuracy?$$\text{Accuracy} = \frac{\text{Correct Predictions}}{\text{Total Predictions}} = \frac{9,900}{10,000} = 99\%$$

An accuracy of 99% sounds incredible! But your model is completely useless because it failed to catch a single instance of fraud. This proves why we need more advanced evaluation tools.

### 2. The Confusion Matrix

To truly understand where a classifier succeeds and fails, we use a Confusion Matrix. A confusion matrix is a table that breaks down actual target labels against what your model predicted.

For a binary classification task (Positive vs. Negative), the matrix contains four key components:
- True Positive (TP): The model predicted Positive, and the true label was actually Positive (e.g., correctly caught fraud).
- True Negative (TN): The model predicted Negative, and the true label was actually Negative (e.g., correctly identified a normal transaction).
- False Positive (FP - Type I Error): The model predicted Positive, but the true label was actually Negative (e.g., flagging a legitimate customer's purchase as fraud). Also known as a false alarm.
- False Negatives (FN - Type II Error): The model predicted Negative, but the true label was actually Positive (e.g., missing an actual fraudulent transaction). This is often the most dangerous error.

### 3. The Core Classification Metrics

Using the counts from the confusion matrix, we derive three essential metrics:

**A. Precision (Quality of Positive Predictions)**

Precision answers the question: Out of all the things the model predicted as positive, how many were actually positive?$$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$
- When it matters most: When false positives are extremely costly. For example, if your spam filter has low precision, important work emails get incorrectly routed to the spam folder.

**B. Recall / Sensitivity (Quantity of Actual Positives Caught)**

- Recall answers the question: Out of all the actual positive items in the dataset, how many did the model manage to find?$$\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}$$
- When it matters most: When false negatives are dangerous. For example, in a medical cancer screening app, you want 100% recall so you never miss a sick patient, even if it causes a few false alarms.

**C. F1-Score (The Harmonic Mean)**

Precision and recall are often in tension—as you tune a model to catch more positives (higher recall), you usually trigger more false alarms (lower precision). The F1-Score combines both into a single metric by calculating their harmonic mean:$$\text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

### 4. Evaluating a Classifier in Python (Scikit-Learn)

Let's write a clean Python script using Scikit-learn to generate a confusion matrix and a full classification report.

In [1]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# 1. Simulated Ground Truth Labels (y_true) vs Model Predictions (y_pred)
# (0 = Legitimate, 1 = Fraudulent)
y_true = np.array([0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0])
y_pred = np.array([0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0])  # Model made a couple of errors

# 2. Compute the Confusion Matrix
conf_matrix = confusion_matrix(y_true, y_pred)
print("--- Confusion Matrix ---")
print(conf_matrix)
# Format breakdown:
# [[True Negatives, False Positives],
#  [False Negatives, True Positives]]

# 3. Compute Accuracy
acc = accuracy_score(y_true, y_pred)
print(f"\nOverall Accuracy: {acc * 100:.2f}%\n")

# 4. Generate Scikit-Learn's built-in Classification Report
# (Automatically calculates Precision, Recall, and F1-Score for each class)
report = classification_report(y_true, y_pred, target_names=["Legitimate (0)", "Fraud (1)"])
print("--- Classification Report ---")
print(report)

--- Confusion Matrix ---
[[8 1]
 [1 2]]

Overall Accuracy: 83.33%

--- Classification Report ---
                precision    recall  f1-score   support

Legitimate (0)       0.89      0.89      0.89         9
     Fraud (1)       0.67      0.67      0.67         3

      accuracy                           0.83        12
     macro avg       0.78      0.78      0.78        12
  weighted avg       0.83      0.83      0.83        12



### Guided Question:

Why can relying solely on accuracy be misleading when evaluating a model trained on a heavily imbalanced dataset?

### Answer:

Relying solely on accuracy when evaluating a model trained on a heavily imbalanced dataset leads to the Accuracy Paradox, where a model that possesses zero actual predictive power can appear to have near-perfect performance.

**The Mathematics of the Problem**

Accuracy is defined as the ratio of correct predictions to total predictions:$$\text{Accuracy} = \frac{\text{True Positives (TP)} + \text{True Negatives (TN)}}{\text{TP} + \text{TN} + \text{False Positives (FP)} + \text{False Negatives (FN)}}$$

When a dataset is heavily imbalanced—such as a fraud detection system with a $99\%$ negative class (legitimate transactions) and a $1\%$ positive class (fraudulent transactions)—a naive baseline model (often called a Zero-R classifier) that always predicts the majority class achieves an accuracy score of $99\%$.Despite this high score, the model is completely useless: it catches zero fraud cases ($TP = 0$).

**Why This Is Dangerous in Practice**

- Masks Minority Class Failure: In most imbalanced domains (e.g., rare disease diagnostics, equipment failure prediction, or credit card fraud), the minority class is precisely what you care about detecting. High accuracy creates a false sense of security by hiding high rates of False Negatives (missing the critical events entirely).
- Optimizes for the Wrong Objective: If you use accuracy as the loss function or evaluation metric during hyperparameter tuning, gradient descent or optimization algorithms will naturally favor predicting the majority class to minimize overall error, worsening the model's ability to discriminate the minority class.
- Ignores Asymmetric Costs: In the real world, the cost of a False Negative is often vastly higher than the cost of a False Positive. Missing a rare cancer diagnosis (False Negative) has catastrophic consequences compared to a false alarm that requires a follow-up test (False Positive). Accuracy treats all errors equally.

**Better Evaluation Metrics for Imbalanced Data**

To accurately measure model performance under class imbalance, evaluate using metrics that account for class distribution:
- **Precision:** Out of all positive predictions made, how many were actually positive? ($\frac{TP}{TP + FP}$). Crucial when the cost of False Positives is high.
- **Recall (Sensitivity):** Out of all actual positive instances in the dataset, how many did the model catch? ($\frac{TP}{TP + FN}$). Crucial when the cost of False Negatives is high.
- **F1-Score:** The harmonic mean of precision and recall, providing a balanced single-metric evaluation:$$F_1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$
- **Precision-Recall AUC (PR-AUC):** Unlike ROC-AUC, which can remain deceptively high on imbalanced datasets due to a large number of True Negatives inflating the False Positive Rate denominator, PR-AUC focuses exclusively on the minority class performance and is much more sensitive to data skew.

### Guided Question:

If you are building a security system for airport luggage scanning, would you optimize your model for higher precision or higher recall? Explain your reasoning.

### Answer:

For an airport luggage scanning security system, you must optimize your model heavily for Recall.

**The Core Trade-Off: Asymmetric Costs**

In threat detection systems, the two types of classification errors carry radically different real-world consequences:

- False Negative (Missing a Threat):
    - Consequence: Catastrophic. If the model fails to flag a weapon, explosive device, or contraband, it passes through security unchecked. The potential cost is loss of human lives, mass casualties, and a catastrophic security breach.
- False Positive (False Alarm):
    - Consequence: Operational inconvenience. If a harmless item (such as a dense stack of electronics, organic materials, or a tightly packed book) is flagged as suspicious, the bag is sent for manual secondary inspection by security personnel.

Because the cost of a False Negative is infinitely higher than the cost of a False Positive, the optimization function must prioritize catching every single potential threat, even at the expense of a higher false alarm rate.

**How This Translates to Model Tuning**

- Maximizing Recall ($Sensitivity = \frac{TP}{TP + FN}$): The primary goal is to drive the False Negative rate as close to zero as possible. You want the model to capture every true threat instance present in the baggage stream.
- Managing Precision ($Precision = \frac{TP}{TP + FP}$): While recall is the primary driver, precision cannot be completely ignored. If precision drops too low (e.g., the model flags 95% of all harmless passenger bags as threats), the checkpoint experiences total operational gridlock, requiring manual inspection of nearly every bag, which paralyzes airport throughput.

**Engineering Practice**

In real-world deployment, engineers handle this by lowering the model's classification threshold. By shifting the decision boundary to favor sensitivity, the system ensures that any ambiguous or borderline scan is automatically routed to human TSA/security agents for secondary verification. In high-stakes security domains, the algorithm acts as a aggressive first filter where human-in-the-loop validation resolves the resulting false positives.

### Guided Question:

In the Scikit-learn code snippet above, what does a False Negative represent in the context of the confusion matrix output?

### Answer:

In the context of this code snippet, a False Negative (FN) represents an instance where the model predicted a transaction was legitimate (0), but the transaction was actually fraudulent (1).Breakdown in This Code Context
- True Class ($y\_true$): 1 (Fraud)
- Predicted Class ($y\_pred$): 0 (Legitimate)

In the confusion matrix layout provided in the script:$$\begin{bmatrix} \text{TN} & \text{FP} \\ \text{FN} & \text{TP} \end{bmatrix}$$

The False Negative count sits at the bottom-left position. It corresponds to a missed fraud case (a Type II error)—the financial system failed to flag a fraudulent transaction, allowing it to go through undetected.